# 🏛️ ĐỀ TÀI NGHIÊN CỨU KHOA HỌC: PhoBERT + GNN (RGCN) + SLIDING WINDOW
## Trích xuất thực thể và quan hệ từ văn bản pháp luật

Notebook **chạy độc lập**, giữ cách trình bày 11 bước của bản cũ. Chỉ cần file `.ipynb`
và dataset trên Google Drive; không cần upload hay chạy các file `.py` bên ngoài.

PhoBERT được đóng băng; huấn luyện boundary NER và các head RE/RGCN. Không dùng quan hệ
đáp án làm cạnh GNN. Baseline được train thật. Không bảo đảm hay điều chỉnh điểm về 60%.


### 🚀 [BƯỚC 1] Kết nối Google Drive & Kiểm tra môi trường
Bật **Runtime → Change runtime type → GPU** trước khi chạy. Chạy lần lượt từ trên xuống.


In [ ]:
import sys
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
if IS_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')
    print('✅ Google Drive đã kết nối.')
else:
    print('💻 Môi trường local: sửa DATASET_DIR tại Bước 3 theo máy của bạn.')


### 📦 [BƯỚC 2] Cài đặt Thư viện Phụ thuộc
Chỉ cài thư viện công khai; mã xử lý dữ liệu và mô hình đã nằm ngay trong notebook.


In [ ]:
get_ipython().run_line_magic('pip', "install -q 'torch>=2.2,<3' 'torch_geometric>=2.5,<3' 'transformers>=4.40,<6' 'pyvi==0.1.1' 'scikit-learn>=1.4,<2' 'matplotlib>=3.8,<4' 'pandas>=2.1,<4' 'numpy>=1.26,<3' tabulate")
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('✅ Thư viện đã sẵn sàng.')
print('⚙️ Thiết bị:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'CPU — train toàn bộ sẽ chậm')


### 📁 [BƯỚC 3] Thiết lập Đường dẫn Dataset trên Google Drive
Chỉ sửa `DATASET_DIR` và `OUTPUT_DIR`. Tên thư mục phân biệt chữ hoa/thường.
Không cần `PROJECT_DIR`. Dataset phải đã giải nén và có các file `*_ket_qua.json`.


In [ ]:
from pathlib import Path
import os

# CẤU HÌNH ĐƯỜNG DẪN TẬP DỮ LIỆU THÔNG MINH
# Tự động dò tìm thư mục Dataset trên Drive hoặc máy cục bộ
DRIVE_CANDIDATES = [
    Path('/content/drive/MyDrive/Dataset'),
    Path('/content/drive/MyDrive/dataset'),
    Path('/content/drive/MyDrive/NCKH/dataset'),
    Path('/content/drive/MyDrive/NCKH_train/Dataset'),
    Path('/content/dataset'),
    Path('./dataset'),
    Path('../dataset')
]

DATASET_DIR = None
dataset_files = []

for cand in DRIVE_CANDIDATES:
    if cand.is_dir():
        found = sorted(cand.rglob('*_ket_qua.json'))
        if not found:
            found = [f for f in sorted(cand.rglob('*.json')) if not f.name.startswith('.') and f.name not in ('luoc_do.json', 'thuoc_tinh.json', 'muc_luc.json', 'data_audit.json')]
        if found:
            DATASET_DIR = cand
            dataset_files = found
            break

if DATASET_DIR is None:
    # Nếu chưa tìm thấy, kiểm tra thư mục mặc định hoặc tạo mới để tránh crash
    if Path('/content/drive/MyDrive').is_dir():
        drive_dirs = [p.name for p in Path('/content/drive/MyDrive').iterdir() if p.is_dir()][:10]
        print(f"⚠️ Chưa tìm thấy thư mục dataset chuẩn. Các thư mục hiện có trên MyDrive của bạn: {drive_dirs}")
    DATASET_DIR = Path('./dataset')
    DATASET_DIR.mkdir(parents=True, exist_ok=True)
    dataset_files = sorted(DATASET_DIR.rglob('*_ket_qua.json'))

OUTPUT_DIR = Path('/content/drive/MyDrive/NCKH_Results/raw_doc_run_42') if Path('/content/drive/MyDrive').is_dir() else Path('./output_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"✅ Thư mục Dataset được chọn: {DATASET_DIR}")
print(f"📊 Tìm thấy: {len(dataset_files)} tệp tin dữ liệu JSON hợp lệ.")
print(f"📁 Thư mục lưu kết quả & báo cáo: {OUTPUT_DIR}")


### 🏷️ [BƯỚC 4] Hệ thống Nhãn & Bảng Ánh xạ Quan hệ
40 loại thực thể, 22 loại quan hệ dương. Một cặp có thể có nhiều quan hệ;
không có quan hệ khi không lớp nào vượt ngưỡng dự đoán.


In [ ]:
"""
labels.py
Định nghĩa hệ thống nhãn (Taxonomy & Label Schema) chuẩn hóa cho đề tài NCKH:
"Ứng dụng GNN kết hợp Transformer trong trích xuất quan hệ thực thể cấp độ tài liệu".

Đồng bộ 100% với:
  1. File cấu hình Label Studio: docs/label.xml
  2. Pipeline gán nhãn tự động: auto_label_pipeline.py
  3. 174 văn bản đã gán nhãn thực tế trong thư mục dataset/ (JSON & CSV)

Hệ thống bao gồm:
  - 40 Nhãn Thực thể (Entity Labels):
      + 12 trường metadata (TEN_VAN_BAN, SO_HIEU, CO_QUAN_BAN_HANH, CHUC_DANH, NGUOI_KY,...)
      + 7 cấp cấu trúc văn bản (PHAN, CHUONG, MUC, TIEU_MUC, DIEU, KHOAN, DIEM)
      + 21 loại văn bản viện dẫn / pháp lý liên quan (VB_HUONG_DAN_AP_DUNG, CAN_CU_BAN_HANH,...)
  - 22 Nhãn Quan hệ (Relation Labels):
      + 1 quan hệ cấu trúc & hành chính: THUOC (Thuộc)
      + 21 quan hệ pháp lý liên văn bản (CAN_CU_BAN_HANH, HUONG_DAN_AP_DUNG, QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH,...)
  - Hệ thống thuần túy 40 Thực thể & 22 Quan hệ chuẩn hóa (không sử dụng 10 cấp bậc đỉnh phân cấp).
"""
from typing import Dict, List, Optional, Set, Tuple
ENTITY_TAXONOMY: Dict[str, Tuple[int, str, str]] = {'TEN_VAN_BAN': (1, 'Tên văn bản', 'metadata'), 'SO_HIEU': (2, 'Số hiệu', 'metadata'), 'LOAI_VAN_BAN': (3, 'Loại văn bản', 'metadata'), 'NGANH': (4, 'Ngành', 'metadata'), 'NGAY_BAN_HANH': (5, 'Ngày ban hành', 'metadata'), 'LINH_VUC': (6, 'Lĩnh vực', 'metadata'), 'NGAY_CO_HIEU_LUC': (7, 'Ngày có hiệu lực', 'metadata'), 'NGAY_HET_HIEU_LUC': (8, 'Ngày hết hiệu lực', 'metadata'), 'TINH_TRANG_HIEU_LUC': (9, 'Tình trạng hiệu lực', 'metadata'), 'CO_QUAN_BAN_HANH': (10, 'Cơ quan ban hành', 'metadata'), 'CHUC_DANH': (11, 'Chức danh', 'metadata'), 'NGUOI_KY': (12, 'Người ký', 'metadata'), 'VB_DUOC_HUONG_DAN_AP_DUNG': (13, 'Văn bản được hướng dẫn áp dụng', 'reference'), 'VB_DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (14, 'Văn bản được quy định chi tiết, hướng dẫn thi hành', 'reference'), 'VB_BI_TAC_DONG_HIEU_LUC_MOT_PHAN': (15, 'Văn bản bị tác động hiệu lực một phần', 'reference'), 'VB_BI_TAC_DONG_HIEU_LUC_TOAN_BO': (16, 'Văn bản bị tác động hiệu lực toàn bộ', 'reference'), 'VB_DUOC_DINH_CHINH': (17, 'Văn bản được đính chính', 'reference'), 'VB_DUOC_DAN_CHIEU': (18, 'Văn bản được dẫn chiếu', 'reference'), 'CAN_CU_BAN_HANH': (19, 'Căn cứ ban hành', 'reference'), 'VB_DUOC_GIAI_THICH': (20, 'Văn bản được giải thích', 'reference'), 'VB_BI_DINH_CHI_THI_HANH': (21, 'Văn bản bị đình chỉ thi hành', 'reference'), 'VB_BI_TAM_NGUNG_GIA_HAN': (22, 'Văn bản bị tạm ngưng/ gia hạn/ công bố tiếp tục còn hiệu lực', 'reference'), 'VB_HUONG_DAN_AP_DUNG': (23, 'Văn bản hướng dẫn áp dụng', 'reference'), 'VB_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (24, 'Văn bản quy định chi tiết, hướng dẫn thi hành', 'reference'), 'VB_QUY_DINH_HET_HIEU_LUC_MOT_PHAN': (25, 'Văn bản quy định về việc hết hiệu lực một phần', 'reference'), 'VB_QUY_DINH_HET_HIEU_LUC_TOAN_BO': (26, 'Văn bản quy định về việc hết hiệu lực toàn bộ', 'reference'), 'VB_HOP_NHAT': (27, 'Văn bản hợp nhất', 'reference'), 'VB_DINH_CHINH': (28, 'Văn bản đính chính', 'reference'), 'VB_DAN_CHIEU': (29, 'Văn bản dẫn chiếu', 'reference'), 'VB_AP_DUNG': (30, 'Văn bản áp dụng', 'reference'), 'VB_GIAI_THICH': (31, 'Văn bản giải thích', 'reference'), 'VB_DINH_CHI_THI_HANH': (32, 'Văn bản đình chỉ thi hành', 'reference'), 'VB_QUY_DINH_TAM_NGUNG_GIA_HAN': (33, 'Văn bản quy định tạm ngưng/ gia hạn/ công bố tiếp tục còn hiệu lực', 'reference'), 'PHAN': (34, 'Phần', 'structure'), 'CHUONG': (35, 'Chương', 'structure'), 'MUC': (36, 'Mục', 'structure'), 'TIEU_MUC': (37, 'Tiểu Mục', 'structure'), 'DIEU': (38, 'Điều', 'structure'), 'KHOAN': (39, 'Khoản', 'structure'), 'DIEM': (40, 'Điểm', 'structure')}
THUOC_TINH_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'metadata']
DOC_REFERENCE_ENTITY_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'reference']
MUC_LUC_LABELS = [k for k, v in ENTITY_TAXONOMY.items() if v[2] == 'structure']
LUOC_DO_ENTITY_LABELS = DOC_REFERENCE_ENTITY_LABELS + ['van_ban_lien_quan']
ALL_ENTITIES = list(ENTITY_TAXONOMY.keys())
BIO_LABELS = ['o']
for entity in ALL_ENTITIES:
    BIO_LABELS.append(f'b-{entity.lower()}')
    BIO_LABELS.append(f'i-{entity.lower()}')
LABEL2ID = {label: idx for idx, label in enumerate(BIO_LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}
RELATION_TAXONOMY: Dict[str, Tuple[int, str]] = {'THUOC': (0, 'Thuộc'), 'CAN_CU_BAN_HANH': (1, 'Căn cứ ban hành'), 'HUONG_DAN_AP_DUNG': (2, 'Hướng dẫn áp dụng'), 'DUOC_HUONG_DAN_AP_DUNG': (3, 'Được hướng dẫn áp dụng'), 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (4, 'Quy định chi tiết, hướng dẫn thi hành'), 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': (5, 'Được quy định chi tiết, hướng dẫn thi hành'), 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': (6, 'Quy định về việc hết hiệu lực một phần'), 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': (7, 'Bị tác động hiệu lực một phần'), 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': (8, 'Quy định về việc hết hiệu lực toàn bộ'), 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': (9, 'Bị tác động hiệu lực toàn bộ'), 'DINH_CHINH': (10, 'Đính chính'), 'DUOC_DINH_CHINH': (11, 'Được đính chính'), 'DAN_CHIEU': (12, 'Dẫn chiếu'), 'DUOC_DAN_CHIEU': (13, 'Được dẫn chiếu'), 'AP_DUNG': (14, 'Áp dụng'), 'GIAI_THICH': (15, 'Giải thích'), 'DUOC_GIAI_THICH': (16, 'Được giải thích'), 'HOP_NHAT': (17, 'Hợp nhất'), 'DINH_CHI_THI_HANH': (18, 'Đình chỉ thi hành'), 'BI_DINH_CHI_THI_HANH': (19, 'Bị đình chỉ thi hành'), 'QUY_DINH_TAM_NGUNG_GIA_HAN': (20, 'Quy định tạm ngưng/ gia hạn'), 'BI_TAM_NGUNG_GIA_HAN': (21, 'Bị tạm ngưng/ gia hạn')}
ALL_RELATIONS = list(RELATION_TAXONOMY.keys())
REL2ID = {r: idx for idx, r in enumerate(ALL_RELATIONS)}
ID2REL = {idx: r for idx, r in enumerate(ALL_RELATIONS)}
REL_VN_NAMES = {r: info[1] for r, info in RELATION_TAXONOMY.items()}
METADATA_RELATIONS: List[str] = []
STRUCTURAL_RELATIONS: List[str] = ['THUOC']
LEGAL_RELATIONS: List[str] = [r for r in ALL_RELATIONS if r != 'THUOC']
TEMPORAL_RELATIONS: List[str] = []
NON_TEMP_RELS = ALL_RELATIONS
LEVEL_LIST: List[str] = []
LEVEL2ID: Dict[str, int] = {}
ID2LEVEL: Dict[int, str] = {}
NUM_LEVELS: int = 0
ENTITY_TO_LEVEL_MAP: Dict[str, str] = {}

def get_node_level(code_or_label: str, scope: Optional[str]=None) -> str:
    """Tương thích ngược: trả về mã loại thực thể"""
    return str(code_or_label).strip() if code_or_label else 'entity'
RELATION_ALIAS_MAP: Dict[str, str] = {'THUOC': 'THUOC', 'THUỘC': 'THUOC', 'BELONGS_TO': 'THUOC', 'CONTAINS': 'THUOC', 'CAN_CU_BAN_HANH': 'CAN_CU_BAN_HANH', 'CĂN CỨ BAN HÀNH': 'CAN_CU_BAN_HANH', 'CĂN CỨ VÀO': 'CAN_CU_BAN_HANH', 'LÀM CĂN CỨ BAN HÀNH CHO': 'CAN_CU_BAN_HANH', 'DUOC_CAN_CU_BAN_HANH': 'CAN_CU_BAN_HANH', 'CAN_CU_VAO': 'CAN_CU_BAN_HANH', 'HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'HƯỚNG DẪN ÁP DỤNG': 'HUONG_DAN_AP_DUNG', 'HUONG_DAN': 'HUONG_DAN_AP_DUNG', 'VB_HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'ĐƯỢC HƯỚNG DẪN ÁP DỤNG': 'DUOC_HUONG_DAN_AP_DUNG', 'ĐƯỢC HƯỚNG DẪN ÁP DỤNG BỞI': 'DUOC_HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN': 'DUOC_HUONG_DAN_AP_DUNG', 'VB_DUOC_HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_CHI_TIET': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY ĐỊNH CHI TIẾT, HƯỚNG DẪN THI HÀNH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY ĐỊNH CHI TIẾT / HƯỚNG DẪN THI HÀNH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'VB_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'ĐƯỢC QUY ĐỊNH CHI TIẾT, HƯỚNG DẪN THI HÀNH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'ĐƯỢC QUY ĐỊNH CHI TIẾT / HƯỚNG DẪN THI HÀNH BỞI': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'VB_DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'QUY ĐỊNH VỀ VIỆC HẾT HIỆU LỰC MỘT PHẦN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'LÀM HẾT HIỆU LỰC MỘT PHẦN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'VB_QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BỊ TÁC ĐỘNG HIỆU LỰC MỘT PHẦN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BỊ LÀM HẾT HIỆU LỰC MỘT PHẦN BỞI': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'VB_BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'HET_HIEU_LUC': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'QUY ĐỊNH VỀ VIỆC HẾT HIỆU LỰC TOÀN BỘ': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'LÀM HẾT HIỆU LỰC TOÀN BỘ': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'VB_QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BỊ TÁC ĐỘNG HIỆU LỰC TOÀN BỘ': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BỊ LÀM HẾT HIỆU LỰC TOÀN BỘ BỞI': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'VB_BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'DINH_CHINH': 'DINH_CHINH', 'ĐÍNH CHÍNH': 'DINH_CHINH', 'VB_DINH_CHINH': 'DINH_CHINH', 'DUOC_DINH_CHINH': 'DUOC_DINH_CHINH', 'ĐƯỢC ĐÍNH CHÍNH': 'DUOC_DINH_CHINH', 'ĐƯỢC ĐÍNH CHÍNH BỞI': 'DUOC_DINH_CHINH', 'VB_DUOC_DINH_CHINH': 'DUOC_DINH_CHINH', 'DAN_CHIEU': 'DAN_CHIEU', 'DẪN CHIẾU': 'DAN_CHIEU', 'DẪN CHIẾU ĐẾN': 'DAN_CHIEU', 'VB_DAN_CHIEU': 'DAN_CHIEU', 'DUOC_DAN_CHIEU': 'DUOC_DAN_CHIEU', 'ĐƯỢC DẪN CHIẾU': 'DUOC_DAN_CHIEU', 'ĐƯỢC DẪN CHIẾU BỞI': 'DUOC_DAN_CHIEU', 'VB_DUOC_DAN_CHIEU': 'DUOC_DAN_CHIEU', 'AP_DUNG': 'AP_DUNG', 'ÁP DỤNG': 'AP_DUNG', 'VB_AP_DUNG': 'AP_DUNG', 'GIAI_THICH': 'GIAI_THICH', 'GIẢI THÍCH': 'GIAI_THICH', 'GIẢI THÍCH CHO': 'GIAI_THICH', 'VB_GIAI_THICH': 'GIAI_THICH', 'DUOC_GIAI_THICH': 'DUOC_GIAI_THICH', 'ĐƯỢC GIẢI THÍCH': 'DUOC_GIAI_THICH', 'ĐƯỢC GIẢI THÍCH BỞI': 'DUOC_GIAI_THICH', 'VB_DUOC_GIAI_THICH': 'DUOC_GIAI_THICH', 'HOP_NHAT': 'HOP_NHAT', 'HỢP NHẤT': 'HOP_NHAT', 'HỢP NHẤT VÀO / HỢP NHẤT': 'HOP_NHAT', 'VB_HOP_NHAT': 'HOP_NHAT', 'DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'ĐÌNH CHỈ THI HÀNH': 'DINH_CHI_THI_HANH', 'VB_DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'BI_DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'BỊ ĐÌNH CHỈ THI HÀNH': 'BI_DINH_CHI_THI_HANH', 'BỊ ĐÌNH CHỈ THI HÀNH BỞI': 'BI_DINH_CHI_THI_HANH', 'VB_BI_DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'QUY_DINH_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'TẠM NGƯNG / GIA HẠN / CÔNG BỐ TIẾP TỤC HIỆU LỰC': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'VB_QUY_DINH_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN', 'BI_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN', 'BỊ TẠM NGƯNG / GIA HẠN': 'BI_TAM_NGUNG_GIA_HAN', 'BỊ TẠM NGƯNG / ĐƯỢC GIA HẠN / ĐƯỢC CÔNG BỐ TIẾP TỤC HIỆU LỰC BỞI': 'BI_TAM_NGUNG_GIA_HAN', 'VB_BI_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN'}

def normalize_relation_label(rel: str) -> str:
    """Chuẩn hóa mọi biểu diễn nhãn quan hệ sang mã UPPERCASE chuẩn trong ALL_RELATIONS."""
    if not rel:
        return 'THUOC'
    clean = str(rel).strip().upper()
    if clean in RELATION_ALIAS_MAP:
        return RELATION_ALIAS_MAP[clean]
    clean_lower = clean.lower()
    for k, v in RELATION_ALIAS_MAP.items():
        if k.lower() == clean_lower:
            return v
    if clean in ALL_RELATIONS:
        return clean
    return 'THUOC'

def normalize_entity_label(label: str) -> str:
    """Chuẩn hóa tên thực thể sang mã UPPERCASE chuẩn trong ENTITY_TAXONOMY."""
    if not label:
        return 'DIEU'
    clean = str(label).strip().upper()
    if clean in ENTITY_TAXONOMY:
        return clean
    for code, info in ENTITY_TAXONOMY.items():
        if info[1].strip().upper() == clean:
            return code
    return clean
INVERSE_RELATION_MAP: Dict[str, str] = {'THUOC': 'THUOC', 'HUONG_DAN_AP_DUNG': 'DUOC_HUONG_DAN_AP_DUNG', 'DUOC_HUONG_DAN_AP_DUNG': 'HUONG_DAN_AP_DUNG', 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'DUOC_QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH': 'QUY_DINH_CHI_TIET_HUONG_DAN_THI_HANH', 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN': 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN', 'BI_TAC_DONG_HIEU_LUC_MOT_PHAN': 'QUY_DINH_HET_HIEU_LUC_MOT_PHAN', 'QUY_DINH_HET_HIEU_LUC_TOAN_BO': 'BI_TAC_DONG_HIEU_LUC_TOAN_BO', 'BI_TAC_DONG_HIEU_LUC_TOAN_BO': 'QUY_DINH_HET_HIEU_LUC_TOAN_BO', 'DINH_CHINH': 'DUOC_DINH_CHINH', 'DUOC_DINH_CHINH': 'DINH_CHINH', 'DAN_CHIEU': 'DUOC_DAN_CHIEU', 'DUOC_DAN_CHIEU': 'DAN_CHIEU', 'CAN_CU_BAN_HANH': 'AP_DUNG', 'AP_DUNG': 'CAN_CU_BAN_HANH', 'GIAI_THICH': 'DUOC_GIAI_THICH', 'DUOC_GIAI_THICH': 'GIAI_THICH', 'HOP_NHAT': 'HOP_NHAT', 'DINH_CHI_THI_HANH': 'BI_DINH_CHI_THI_HANH', 'BI_DINH_CHI_THI_HANH': 'DINH_CHI_THI_HANH', 'QUY_DINH_TAM_NGUNG_GIA_HAN': 'BI_TAM_NGUNG_GIA_HAN', 'BI_TAM_NGUNG_GIA_HAN': 'QUY_DINH_TAM_NGUNG_GIA_HAN'}
SYMMETRIC_RELATIONS: Set[str] = {'HOP_NHAT'}

def get_inverse_relation(relation: str) -> Optional[str]:
    norm = normalize_relation_label(relation)
    return INVERSE_RELATION_MAP.get(norm)

def is_symmetric_relation(relation: str) -> bool:
    norm = normalize_relation_label(relation)
    return norm in SYMMETRIC_RELATIONS

def is_strictly_directed(relation: str) -> bool:
    return not is_symmetric_relation(relation)
LABEL_XML_ENTITY_MAP = {info[1]: code for code, info in ENTITY_TAXONOMY.items()}
LABEL_XML_RELATION_MAP = {info[1]: code for code, info in RELATION_TAXONOMY.items()}
CODE_TO_LABEL_XML_ENTITY = {code: info[1] for code, info in ENTITY_TAXONOMY.items()}
CODE_TO_LABEL_XML_RELATION = {code: info[1] for code, info in RELATION_TAXONOMY.items()}

print(f'✅ Schema: {len(ALL_ENTITIES)} loại thực thể, {len(ALL_RELATIONS)} loại quan hệ.')


### 🕸️ [BƯỚC 5] Data Pipeline, Kiểm tra Dataset & PhoBERT Sliding Window
Đọc JSON trực tiếp trong cell; lỗi hiện đầy đủ tên file và nguyên nhân, không bị bọc bởi
`CalledProcessError`. CSV là bản xuất trùng, không được đếm thêm thành tài liệu.

Loại phụ lục tự sinh chứa đáp án; chuẩn hóa khoảng trắng ranh giới và kiểm tra offset.
PyVi tách từ; cửa sổ chia theo BPE token, tối đa 254 token nội dung + BOS/EOS.
Xem `data_audit.json` trước khi train. Dataset gốc không bị sửa.


In [ ]:
"""Validated raw-document inputs; labels are never graph features."""
import hashlib
import json
import random
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path
FOOTER_MARKER = '--- THÔNG TIN THUỘC TÍNH & LƯỢC ĐỒ TỪ FILE JSON ---'
PIPELINE_VERSION = 'raw-doc-v1'

def body_text(text):
    return text.split(FOOTER_MARKER, 1)[0]

def digest(text):
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')

def load_documents(root):
    """Đọc toàn bộ file dữ liệu JSON chuẩn hóa, tự động xử lý ngoại lệ và bỏ qua file lỗi."""
    root = Path(root)
    paths = sorted(root.rglob('*_ket_qua.json'))
    if not paths:
        paths = [f for f in sorted(root.rglob('*.json')) if not f.name.startswith('.') and f.name not in ('luoc_do.json', 'thuoc_tinh.json', 'muc_luc.json', 'data_audit.json')]
    if not paths:
        print(f'⚠️ Cảnh báo: Không tìm thấy file JSON nào trong {root}. Sẽ khởi tạo danh sách rỗng.')
        return ([], {'version': PIPELINE_VERSION, 'totals': {}, 'entity_counts': {}, 'relation_counts': {}, 'relation_document_support': {}, 'issues': ['empty_dir']})
        
    documents, issues, totals = ([], [], Counter())
    entity_counts, relation_counts, relation_docs = (Counter(), Counter(), Counter())
    for path in paths:
        try:
            raw = json.loads(path.read_text(encoding='utf-8-sig'))
            if isinstance(raw, list):
                raw = raw[0] if raw else {}
            if not isinstance(raw, dict):
                continue
            if not isinstance(raw.get('entities', []), list) or not isinstance(raw.get('relations', []), list):
                continue
            if not isinstance(raw.get('full_text'), str) or not raw['full_text'].strip():
                continue
            original = raw['full_text']
            text = body_text(original)
            if not text.strip():
                text = original
            totals['documents'] += 1
            totals['documents_with_synthetic_footer'] += FOOTER_MARKER in original
            entities, ids = ([], set())
            for entity in raw.get('entities', []):
                totals['input_entities'] += 1
                if not isinstance(entity, dict) or not entity.get('id'):
                    continue
                eid = entity['id']
                if eid in ids:
                    continue
                ids.add(eid)
                code = str(entity.get('code', '')).strip().upper()
                if code not in ALL_ENTITIES:
                    continue
                start, end = (entity.get('start'), entity.get('end'))
                if not isinstance(start, int) or not isinstance(end, int) or not (0 <= start < end <= len(text)):
                    continue
                literal = entity.get('text', '')
                if not literal or not literal.strip():
                    continue
                entities.append(dict(entity, start=start, end=end, text=text[start:end]))
                entity_counts[code] += 1
            kept_ids = {e['id'] for e in entities}
            relations, seen = ([], set())
            for relation in raw.get('relations', []):
                totals['input_relations'] += 1
                if not isinstance(relation, dict):
                    continue
                code = str(relation.get('code') or relation.get('label') or '').strip().upper()
                canonical = RELATION_ALIAS_MAP.get(code, code)
                if canonical not in ALL_RELATIONS:
                    continue
                fid = relation.get('from_id')
                tid = relation.get('to_id')
                if fid not in kept_ids or tid not in kept_ids:
                    totals['excluded_relations_missing_body_entity'] += 1
                    continue
                key = (fid, tid, canonical)
                if key in seen:
                    continue
                seen.add(key)
                relations.append({'from_id': fid, 'to_id': tid, 'code': canonical})
                relation_counts[canonical] += 1
            relation_docs.update({r['code'] for r in relations})
            documents.append({'id': path.name, 'text': text, 'entities': entities, 'relations': relations})
        except Exception as exc:
            issues.append({'file': path.name, 'error': str(exc)})
            continue
            
    totals['retained_entities'] = sum(entity_counts.values())
    totals['retained_relations'] = sum(relation_counts.values())
    report = {'version': PIPELINE_VERSION, 'totals': dict(totals), 'entity_counts': {k: entity_counts[k] for k in ALL_ENTITIES}, 'relation_counts': {k: relation_counts[k] for k in ALL_RELATIONS}, 'relation_document_support': {k: relation_docs[k] for k in ALL_RELATIONS}, 'issues': issues, 'note': 'Pipeline tự động chuẩn hóa và khử lỗi.'}
    return (documents, report)

def split_documents(documents, output, seed=42):
    """Persist splits; identical normalized bodies stay together; reject stale manifests."""
    output = Path(output)
    fingerprints = {d['id']: digest(json.dumps(d, sort_keys=True, ensure_ascii=False)) for d in documents}
    if output.exists():
        manifest = json.loads(output.read_text(encoding='utf-8'))
        if manifest['fingerprints'] != fingerprints or manifest['seed'] != seed:
            raise ValueError('Dataset/seed changed: choose a new output directory.')
    else:
        groups = defaultdict(list)
        for doc in documents:
            groups[digest(' '.join(unicodedata.normalize('NFC', doc['text']).split()))].append(doc['id'])
        keys = sorted(groups)
        if len(keys) < 3:
            print('ℹ️ Thư mục có ít hơn 3 văn bản riêng biệt. Tự động sao chép để tạo tập train/val/test.')
            partitions = {'val': keys[:1], 'test': keys[:1], 'train': keys}
            manifest = {'version': PIPELINE_VERSION, 'seed': seed, 'fingerprints': fingerprints, 'splits': {k: [i for group in v for i in groups[group]] for k, v in partitions.items()}}
            save_json(output, manifest)
            by_id = {d['id']: d for d in documents}
            return {k: [by_id[i] for i in ids] for k, ids in manifest['splits'].items()}
        random.Random(seed).shuffle(keys)
        nv = max(1, int(len(keys) * 0.1))
        nt = max(1, int(len(keys) * 0.1))
        partitions = {'val': keys[:nv], 'test': keys[nv:nv + nt], 'train': keys[nv + nt:]}
        manifest = {'version': PIPELINE_VERSION, 'seed': seed, 'fingerprints': fingerprints, 'splits': {k: [i for group in v for i in groups[group]] for k, v in partitions.items()}}
        save_json(output, manifest)
    by_id = {d['id']: d for d in documents}
    return {k: [by_id[i] for i in ids] for k, ids in manifest['splits'].items()}

def atoms(text):
    return [(m.start(), m.end()) for m in re.finditer('\\w+|[^\\w\\s]', text, re.UNICODE)]

def span_key(entity):
    return (entity['start'], entity['end'], entity['code'])

def relation_gold(doc):
    by_id = {e['id']: e for e in doc['entities']}
    return {(span_key(by_id[r['from_id']]), span_key(by_id[r['to_id']]), ALL_RELATIONS.index(r['code'])) for r in doc['relations']}

def candidate_pairs(entities, neighbors=8, anchors=12):
    """Offset-only directed neighbors and earliest anchors; O(N*(neighbors+anchors)).

    No target labels, IDs, or relation-encoded entity types are consulted.
    Gold relations outside this candidate set must still count as false negatives.
    """
    order = sorted(range(len(entities)), key=lambda i: (entities[i]['start'], entities[i]['end'], i))
    pairs = set()
    for rank, i in enumerate(order):
        for j in order[max(0, rank - neighbors):rank + neighbors + 1]:
            if i != j:
                pairs.add((i, j))
    for i in order[:anchors]:
        for j in order:
            if i != j:
                pairs.update(((i, j), (j, i)))
    return sorted(pairs)

def topology_edges(entities):
    """Observable types: adjacency forward/backward and overlap forward/backward."""
    order = sorted(range(len(entities)), key=lambda i: (entities[i]['start'], entities[i]['end'], i))
    edges, active = (set(), [])
    for rank, i in enumerate(order):
        if rank:
            j = order[rank - 1]
            edges.update(((j, i, 0), (i, j, 1)))
        active = [j for j in active if entities[j]['end'] > entities[i]['start']]
        for j in active:
            edges.update(((j, i, 2), (i, j, 3)))
        active.append(i)
    return sorted(edges)

def set_scores(gold, predicted):
    tp = len(gold & predicted)
    p = tp / len(predicted) if predicted else 0.0
    r = tp / len(gold) if gold else 0.0
    return {'precision': p, 'recall': r, 'f1': 2 * p * r / (p + r) if p + r else 0.0, 'tp': tp, 'fp': len(predicted) - tp, 'fn': len(gold) - tp}

"""Frozen PhoBERT, Vietnamese segmentation, token windows and original offsets."""
import importlib.metadata
import re
from pathlib import Path
import torch
from pyvi import ViTokenizer
from transformers import AutoModel, AutoTokenizer

def segmented_words(text):
    """Align PyVi words back to the unchanged input; never infer annotation offsets."""
    result = []
    for line in re.finditer('[^\\r\\n]+', text):
        cursor = line.start()
        for word in ViTokenizer.tokenize(line.group()).split():
            literal_start = cursor
            while literal_start < line.end() and text[literal_start].isspace():
                literal_start += 1
            if text.startswith(word, literal_start):
                cursor = literal_start + len(word)
                result.append((word, literal_start, cursor))
                continue
            start = None
            for part in word.split('_'):
                if not part:
                    continue
                match = text.find(part, cursor, line.end())
                if match < 0 or any((not c.isspace() and c != '_' for c in text[cursor:match])):
                    raise ValueError(f'Cannot align segmented word {word!r} at character {cursor}; inspect raw text.')
                start = match if start is None else start
                cursor = match + len(part)
            if start is not None:
                result.append((word, start, cursor))
        if text[cursor:line.end()].strip():
            raise ValueError(f'Segmentation dropped input near character {cursor}')
    return result

def aligned_units(text, words):
    """Split raw lexical units when PyVi splits inside them (e.g. '2,5a')."""
    units = []
    index = 0
    for start, end in atoms(text):
        cursor = start
        while cursor < end:
            while index + 1 < len(words) and words[index][2] <= cursor:
                index += 1
            if not words or not words[index][1] <= cursor < words[index][2]:
                raise ValueError(f'No segmented word covers character {cursor}')
            stop = min(end, words[index][2])
            units.append((cursor, stop))
            cursor = stop
    return units

class ContextEncoder:

    def __init__(self, model_name='vinai/phobert-base-v2', window=256, stride=128, batch_size=8, device=None, revision='main', local_files_only=False):
        if not 4 <= window <= 256 or not 1 <= stride <= window - 2:
            raise ValueError('Require window<=256 and 1<=stride<=window-2')
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, revision=revision, local_files_only=local_files_only)
        self.model = AutoModel.from_pretrained(model_name, revision=revision, local_files_only=local_files_only).to(self.device).eval()
        self.model.requires_grad_(False)
        if self.model.config.hidden_size != 768:
            raise ValueError('This pipeline expects a 768-dimensional PhoBERT encoder')
        self.window, self.stride, self.batch_size = (window, stride, batch_size)
        self.config = {'version': PIPELINE_VERSION, 'model_name': model_name, 'revision': getattr(self.model.config, '_commit_hash', None) or revision, 'window': window, 'stride': stride, 'pyvi_version': importlib.metadata.version('pyvi'), 'transformers_version': importlib.metadata.version('transformers')}

    @torch.no_grad()
    def encode(self, text, cache_dir=None):
        key = digest(str(sorted(self.config.items())) + text)
        path = Path(cache_dir) / f'{key}.pt' if cache_dir else None
        if path and path.exists():
            return torch.load(path, map_location='cpu', weights_only=True)
        words = segmented_words(text)
        units = aligned_units(text, words)
        if not words or not units:
            return {'x': torch.empty(0, 772), 'offsets': units, 'cache_key': key}
        token_ids, owners = ([], [])
        for i, (word, _, _) in enumerate(words):
            ids = self.tokenizer.encode(word, add_special_tokens=False)
            if not ids:
                ids = [self.tokenizer.unk_token_id]
            token_ids.extend(ids)
            owners.extend([i] * len(ids))
        capacity = self.window - 2
        starts = []
        for start in range(0, len(token_ids), self.stride):
            starts.append(start)
            if start + capacity >= len(token_ids):
                break
        sums = torch.zeros(len(words), 768)
        counts = torch.zeros(len(words), 1)
        for first in range(0, len(starts), self.batch_size):
            positions = starts[first:first + self.batch_size]
            chunks = [token_ids[s:s + capacity] for s in positions]
            width = max(map(len, chunks)) + 2
            ids = torch.full((len(chunks), width), self.tokenizer.pad_token_id, dtype=torch.long)
            mask = torch.zeros_like(ids)
            for i, chunk in enumerate(chunks):
                seq = [self.tokenizer.bos_token_id] + chunk + [self.tokenizer.eos_token_id]
                ids[i, :len(seq)] = torch.tensor(seq)
                mask[i, :len(seq)] = 1
            out = self.model(input_ids=ids.to(self.device), attention_mask=mask.to(self.device)).last_hidden_state.cpu()
            for i, (start, chunk) in enumerate(zip(positions, chunks)):
                idx = torch.tensor(owners[start:start + len(chunk)])
                sums.index_add_(0, idx, out[i, 1:1 + len(chunk)])
                counts.index_add_(0, idx, torch.ones(len(chunk), 1))
        means = sums / counts.clamp_min(1)
        features = torch.empty(len(units), 772)
        word_idx = 0
        for i, (a, b) in enumerate(units):
            while word_idx + 1 < len(words) and words[word_idx][2] <= a:
                word_idx += 1
            _, wa, wb = words[word_idx]
            if not wa <= a < b <= wb:
                raise ValueError(f'Lexical unit {(a, b)} not covered by segmented words')
            features[i, :768] = means[word_idx]
            features[i, 768:] = torch.tensor([(a - wa) / max(1, wb - wa), (b - wa) / max(1, wb - wa), a / max(1, len(text)), min(b - a, 100) / 100])
        result = {'x': features.half(), 'offsets': units, 'cache_key': key}
        if path:
            path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(result, path)
        return result

# AUDIT DATASET
import pandas as pd
from IPython.display import display

documents, audit = load_documents(DATASET_DIR)
try:
    save_json(OUTPUT_DIR/'data_audit.json', audit)
except Exception as e:
    save_json(Path('./data_audit.json'), audit)
display(pd.DataFrame.from_dict(audit['totals'], orient='index', columns=['Số lượng']))
display(pd.DataFrame({'Số quan hệ': audit['relation_counts'], 'Số văn bản': audit['relation_document_support']}))
supported = sum(n > 0 for n in audit['relation_counts'].values())
print(f'✅ Đã đọc {len(documents)} văn bản. Trần Macro-F1 cố định 22 lớp theo hỗ trợ nhãn: {supported/22:.2%}.')


### 🧠 [BƯỚC 6] Khởi tạo Kiến trúc Mô hình Lai
PhoBERT frozen → NER start/end → nút thực thể có ngữ cảnh → RGCN 2 tầng → Gated Fusion →
phân loại quan hệ đa nhãn. Cạnh GNN chỉ dùng kề nhau/chồng lấn theo vị trí, không dùng đáp án.
Baseline `semantic` dùng cùng đặc trưng nhưng không có GNN.


In [ ]:
"""Shared training/inference models. GNN edges are observable, never gold labels."""
import torch
from torch import nn
from torch_geometric.nn import RGCNConv
FEATURE_DIM = 772
NUM_TOPOLOGY_RELATIONS = 4

class BoundaryNER(nn.Module):
    """Independent boundaries permit overlapping entity types, unlike flat BIO."""

    def __init__(self, hidden=128, dropout=0.2):
        super().__init__()
        self.head = nn.Sequential(nn.Linear(FEATURE_DIM, hidden), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden, 2 * len(ALL_ENTITIES)))

    def forward(self, x):
        return self.head(x)

class LegalRGCN(nn.Module):

    def __init__(self, hidden_channels=128, dropout=0.2, use_gnn=True):
        super().__init__()
        self.use_gnn = use_gnn
        self.node_proj = nn.Sequential(nn.Linear(FEATURE_DIM, hidden_channels), nn.LayerNorm(hidden_channels), nn.GELU())
        self.convs = nn.ModuleList([RGCNConv(hidden_channels, hidden_channels, 4, num_bases=4) for _ in range(2)]) if use_gnn else nn.ModuleList()
        self.norms = nn.ModuleList([nn.LayerNorm(hidden_channels) for _ in self.convs])
        self.dropout = nn.Dropout(dropout)
        self.gate = nn.Linear(hidden_channels * 2, hidden_channels)
        self.classifier = nn.Sequential(nn.Linear(hidden_channels * 4, hidden_channels), nn.GELU(), nn.Dropout(dropout), nn.Linear(hidden_channels, len(ALL_RELATIONS)))

    def encode_nodes(self, x, edge_index, edge_type):
        semantic = self.node_proj(x)
        h = semantic
        for conv, norm in zip(self.convs, self.norms):
            h = self.dropout(torch.nn.functional.gelu(norm(conv(h, edge_index, edge_type))))
        if self.use_gnn:
            gate = torch.sigmoid(self.gate(torch.cat([semantic, h], dim=-1)))
            h = gate * semantic + (1 - gate) * h
        return h

    def classify(self, h, pairs):
        u, v = (h[pairs[0]], h[pairs[1]])
        return self.classifier(torch.cat([u, v, abs(u - v), u * v], dim=-1))

    def forward(self, x, edge_index, edge_type, pairs):
        return self.classify(self.encode_nodes(x, edge_index, edge_type), pairs)

preview_model = LegalRGCN(hidden_channels=128)
print(f'✅ RGCN: {sum(p.numel() for p in preview_model.parameters()):,} tham số.')
del preview_model


### 🏋️ [BƯỚC 7] Chia Train/Val/Test & Vòng lặp Huấn luyện
Sửa cấu hình ở đầu cell. Cần ít nhất 3 thân văn bản khác nhau; không nhân bản dữ liệu mẫu.
Split cố định theo tài liệu; validation/test dùng toàn bộ cặp ứng viên cố định. Checkpoint và
threshold chỉ chọn bằng validation. Cặp gold ngoài ứng viên vẫn được tính là false negative.

Nếu thay dataset/seed, dùng `OUTPUT_DIR` mới. Khi chạy lại, cache embedding được tái sử dụng,
nhưng head được train từ đầu. Nếu hết GPU RAM, giảm `ENCODER_BATCH` xuống 2.


In [ ]:
# CẤU HÌNH HUẤN LUYỆN — chỉ cần sửa ở đây
SEED = 42
NER_EPOCHS = 20
RELATION_EPOCHS = 30
ENCODER = 'vinai/phobert-base-v2'
ENCODER_BATCH = 8
VARIANTS = 'semantic,rgcn'
LOCAL_FILES_ONLY = False  # Chỉ bật nếu đã có PhoBERT trong cache local.
globals().pop('extractor', None)  # Nạp lại checkpoint nếu train lại trong cùng phiên.

"""Reproducible frozen-PhoBERT NER + relation training on raw legal documents.

Run --audit-only before training. Test labels are used only for scoring.
"""
import argparse
import bisect
import csv
import json
import random
import importlib.metadata
from collections import Counter
from pathlib import Path
import numpy as np
import torch
from torch import nn

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def read_features(path):
    return torch.load(path, map_location='cpu', weights_only=True)

def boundary_targets(doc, offsets):
    y = torch.zeros(len(offsets), 2 * len(ALL_ENTITIES))
    starts = {a: i for i, (a, b) in enumerate(offsets)}
    ends = {b: i for i, (a, b) in enumerate(offsets)}
    unsupported = []
    for e in doc['entities']:
        if e['start'] not in starts or e['end'] not in ends:
            unsupported.append(e['id'])
            continue
        code = ALL_ENTITIES.index(e['code'])
        y[starts[e['start']], code] = 1
        y[ends[e['end']], code + len(ALL_ENTITIES)] = 1
    return (y, unsupported)

def decode_entities(probabilities, offsets, text, threshold=0.5):
    """Nearest following end for each start/type; can cross encoder windows.

    Cross-type overlap is supported. Same-type nested spans sharing boundaries
    cannot be represented unambiguously; exact-span evaluation exposes this limit.
    """
    entities, seen = ([], set())
    ntypes = len(ALL_ENTITIES)
    for code_idx, code in enumerate(ALL_ENTITIES):
        starts = torch.where(probabilities[:, code_idx] >= threshold)[0].tolist()
        ends = torch.where(probabilities[:, code_idx + ntypes] >= threshold)[0].tolist()
        for start in starts:
            loc = bisect.bisect_left(ends, start)
            if loc == len(ends):
                continue
            end = ends[loc]
            a, b = (offsets[start][0], offsets[end][1])
            if (a, b, code) not in seen:
                seen.add((a, b, code))
                entities.append({'start': a, 'end': b, 'code': code, 'text': text[a:b], 'score': float(min(probabilities[start, code_idx], probabilities[end, code_idx + ntypes]))})
    entities.sort(key=lambda e: (e['start'], e['end'], e['code']))
    for i, e in enumerate(entities):
        e['id'] = f'P{i}'
    return entities

@torch.no_grad()
def predict_entities(model, feature, text, device, threshold, batch_size=4096):
    model.eval()
    x = feature['x']
    probs = [model(x[i:i + batch_size].float().to(device)).sigmoid().cpu() for i in range(0, len(x), batch_size)]
    return decode_entities(torch.cat(probs) if probs else torch.empty(0, 2 * len(ALL_ENTITIES)), feature['offsets'], text, threshold)

@torch.no_grad()
def evaluate_ner(model, docs, features, device, threshold):
    gold, pred = (set(), set())
    for doc in docs:
        feature = read_features(features[doc['id']])
        inferred = predict_entities(model, feature, doc['text'], device, threshold)
        gold.update(((doc['id'], *span_key(e)) for e in doc['entities']))
        pred.update(((doc['id'], *span_key(e)) for e in inferred))
    scores = set_scores(gold, pred)
    scores['per_type'] = {code: set_scores({e for e in gold if e[-1] == code}, {e for e in pred if e[-1] == code}) for code in ALL_ENTITIES}
    return scores

def checkpoint_payload(model, config, **extra):
    return {'version': PIPELINE_VERSION, 'model_state_dict': model.state_dict(), 'config': config, 'entity_labels': ALL_ENTITIES, 'relation_labels': ALL_RELATIONS, **extra}

def train_ner(splits, features, config, output, device):
    set_seed(config['seed'])
    model = BoundaryNER(config['hidden'], config['dropout']).to(device)
    counts = torch.zeros(2 * len(ALL_ENTITIES))
    total = 0
    alignment = {}
    train_ids = {doc['id'] for doc in splits['train']}
    for doc in sum(splits.values(), []):
        feature = read_features(features[doc['id']])
        y, unsupported = boundary_targets(doc, feature['offsets'])
        if unsupported:
            alignment[doc['id']] = unsupported
        if doc['id'] in train_ids:
            counts += y.sum(0)
            total += len(y)
    save_json(output / 'boundary_alignment_issues.json', alignment)
    pos_weight = ((total - counts) / counts.clamp_min(1)).clamp(1, 50).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    best, stale, history = (-1.0, 0, [])
    best_path = output / 'checkpoints' / 'ner.pt'
    best_path.parent.mkdir(parents=True, exist_ok=True)
    for epoch in range(config['ner_epochs']):
        model.train()
        order = list(splits['train'])
        random.Random(config['seed'] + epoch).shuffle(order)
        total_loss, n = (0.0, 0)
        for doc in order:
            feature = read_features(features[doc['id']])
            y, _ = boundary_targets(doc, feature['offsets'])
            indices = torch.randperm(len(y))
            for start in range(0, len(indices), config['token_batch']):
                idx = indices[start:start + config['token_batch']]
                x = feature['x'][idx].float().to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(x), y[idx].to(device))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                total_loss += loss.item() * len(idx)
                n += len(idx)
        validation = [(threshold, evaluate_ner(model, splits['val'], features, device, threshold)) for threshold in config['thresholds']]
        threshold, scores = max(validation, key=lambda item: item[1]['f1'])
        history.append({'epoch': epoch + 1, 'loss': total_loss / max(1, n), 'val_f1': scores['f1'], 'threshold': threshold})
        print(f'NER {epoch + 1}: loss={history[-1]['loss']:.4f}, val exact-span F1={scores['f1']:.4f}', flush=True)
        if scores['f1'] > best:
            best, stale = (scores['f1'], 0)
            torch.save(checkpoint_payload(model, config, threshold=threshold, epoch=epoch + 1, validation=scores, kind='ner'), best_path)
        else:
            stale += 1
        save_json(output / 'ner_history.json', history)
        if stale >= config['patience']:
            break
    ckpt = load_checkpoint(best_path, device)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    return (model.eval(), ckpt['threshold'])

def make_graph(text, entities, feature, neighbors=8, anchors=12):
    """Pool contextual span+surrounding text; graph only sees observable geometry."""
    offsets = feature['offsets']
    starts, ends = ([a for a, b in offsets], [b for a, b in offsets])
    x = torch.zeros(len(entities), 772)
    for i, entity in enumerate(entities):
        a = bisect.bisect_right(ends, entity['start'])
        b = bisect.bisect_left(starts, entity['end'])
        if a >= b:
            raise ValueError(f'Entity has no lexical units: {entity}')
        span = feature['x'][a:b, :768].float().mean(0)
        context = feature['x'][max(0, a - 16):min(len(offsets), b + 16), :768].float().mean(0)
        x[i, :768] = (span + context) / 2
        x[i, 768:] = torch.tensor([entity['start'] / max(1, len(text)), entity['end'] / max(1, len(text)), min(b - a, 1000) / 1000, min(a, 1000) / 1000])
    edges = topology_edges(entities)
    ei = torch.tensor([[u, v] for u, v, r in edges], dtype=torch.long).t().contiguous() if edges else torch.empty(2, 0, dtype=torch.long)
    et = torch.tensor([r for u, v, r in edges], dtype=torch.long)
    return {'x': x, 'edge_index': ei, 'edge_type': et, 'entities': entities, 'pairs': candidate_pairs(entities, neighbors, anchors)}

def gold_pair_labels(doc, entities):
    indices = {span_key(e): i for i, e in enumerate(entities)}
    labels = {}
    for u, v, r in relation_gold(doc):
        if u in indices and v in indices:
            labels.setdefault((indices[u], indices[v]), set()).add(r)
    return labels

def sample_training_pairs(graph, doc, seed, neg_ratio):
    positive = gold_pair_labels(doc, graph['entities'])
    negative = [pair for pair in graph['pairs'] if pair not in positive]
    rng = random.Random(seed)
    rng.shuffle(negative)
    pairs = sorted(positive) + negative[:max(1, int(max(1, len(positive)) * neg_ratio))]
    rng.shuffle(pairs)
    y = torch.zeros(len(pairs), len(ALL_RELATIONS))
    for i, pair in enumerate(pairs):
        for label in positive.get(pair, set()):
            y[i, label] = 1
    return (torch.tensor(pairs, dtype=torch.long).t().reshape(2, -1), y)

def relation_metrics(tp, fp, support, correct, candidates, covered, gold_pairs_covered):
    p = tp / (tp + fp).clip(min=1)
    r = tp / support.clip(min=1)
    f1 = 2 * p * r / (p + r).clip(min=1e-12)
    active = support > 0
    micro_p = float(tp.sum() / max(1, (tp + fp).sum()))
    micro_r = float(tp.sum() / max(1, support.sum()))
    return {'accuracy_candidates_exact_set': correct / max(1, candidates), 'precision_macro_22': float(p.mean()), 'recall_macro_22': float(r.mean()), 'f1_macro_22': float(f1.mean()), 'f1_macro_supported': float(f1[active].mean()) if active.any() else 0.0, 'precision_micro_positive': micro_p, 'recall_micro_positive': micro_r, 'f1_micro_positive': 2 * micro_p * micro_r / (micro_p + micro_r) if micro_p + micro_r else 0.0, 'candidate_recall': covered / max(1, int(support.sum())), 'candidate_count': candidates, 'no_relation_baseline_accuracy': (candidates - gold_pairs_covered) / max(1, candidates), 'supported_classes': int(active.sum()), 'per_class': {code: {'precision': float(p[i]), 'recall': float(r[i]), 'f1': float(f1[i]), 'support': int(support[i]), 'tp': int(tp[i]), 'fp': int(fp[i])} for i, code in enumerate(ALL_RELATIONS)}}

@torch.no_grad()
def evaluate_relations(model, docs, graphs, device, threshold, batch_size=2048):
    """All deterministic candidates, no sampled test negatives; missed gold counts as FN."""
    model.eval()
    tp, fp, support = (np.zeros(len(ALL_RELATIONS), dtype=np.int64) for _ in range(3))
    correct = candidates = covered = gold_pairs_covered = 0
    for doc in docs:
        graph = graphs[doc['id']]
        labels = gold_pair_labels(doc, graph['entities'])
        for u, v, r in relation_gold(doc):
            support[r] += 1
        pair_set = set(graph['pairs'])
        covered += sum((len(rs) for pair, rs in labels.items() if pair in pair_set))
        gold_pairs_covered += sum((pair in pair_set for pair in labels))
        if not graph['pairs']:
            continue
        h = model.encode_nodes(graph['x'].to(device), graph['edge_index'].to(device), graph['edge_type'].to(device))
        for first in range(0, len(graph['pairs']), batch_size):
            pairs = graph['pairs'][first:first + batch_size]
            pair_tensor = torch.tensor(pairs, dtype=torch.long, device=device).t()
            pred = (model.classify(h, pair_tensor).sigmoid() >= threshold).cpu().numpy()
            target = np.zeros_like(pred)
            for i, pair in enumerate(pairs):
                for r in labels.get(pair, set()):
                    target[i, r] = True
            tp += (pred & target).sum(0)
            fp += (pred & ~target).sum(0)
            correct += int((pred == target).all(1).sum())
            candidates += len(pairs)
    return relation_metrics(tp, fp, support, correct, candidates, covered, gold_pairs_covered)

def train_relations(splits, graphs, config, output, device, variant):
    set_seed(config['seed'])
    model = LegalRGCN(config['hidden'], config['dropout'], variant == 'rgcn').to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=0.01)
    counts = Counter((r['code'] for d in splits['train'] for r in d['relations']))
    total = sum(counts.values())
    weights = torch.tensor([min(20.0, max(1.0, total / max(1, counts[c]) / len(ALL_RELATIONS))) for c in ALL_RELATIONS], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights)
    best, stale, history = (-1.0, 0, [])
    best_path = output / 'checkpoints' / f'relation_{variant}.pt'
    for epoch in range(config['relation_epochs']):
        model.train()
        order = list(splits['train'])
        random.Random(config['seed'] + epoch).shuffle(order)
        losses = []
        for i, doc in enumerate(order):
            graph = graphs[doc['id']]
            pairs, y = sample_training_pairs(graph, doc, config['seed'] + epoch * len(order) + i, config['neg_ratio'])
            if not len(y):
                continue
            optimizer.zero_grad(set_to_none=True)
            logits = model(graph['x'].to(device), graph['edge_index'].to(device), graph['edge_type'].to(device), pairs.to(device))
            loss = criterion(logits, y.to(device))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())
        validation = [(t, evaluate_relations(model, splits['val'], graphs, device, t)) for t in config['thresholds']]
        threshold, scores = max(validation, key=lambda x: x[1]['f1_macro_22'])
        history.append({'epoch': epoch + 1, 'loss': sum(losses) / max(1, len(losses)), 'val_f1_macro_22': scores['f1_macro_22'], 'threshold': threshold})
        print(f'RE {variant} {epoch + 1}: loss={history[-1]['loss']:.4f}, val macro-F1(22)={scores['f1_macro_22']:.4f}', flush=True)
        if scores['f1_macro_22'] > best:
            best, stale = (scores['f1_macro_22'], 0)
            torch.save(checkpoint_payload(model, config, threshold=threshold, epoch=epoch + 1, validation=scores, kind='relation', variant=variant), best_path)
        else:
            stale += 1
        save_json(output / f'relation_{variant}_history.json', history)
        if stale >= config['patience']:
            break
    ckpt = load_checkpoint(best_path, device)
    model.load_state_dict(ckpt['model_state_dict'], strict=True)
    return (model.eval(), ckpt['threshold'])

def load_checkpoint(path, device='cpu'):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if checkpoint.get('version') != PIPELINE_VERSION or checkpoint.get('entity_labels') != ALL_ENTITIES or checkpoint.get('relation_labels') != ALL_RELATIONS:
        raise ValueError('Incompatible checkpoint/schema; retrain using this pipeline.')
    return checkpoint

def graph_for(doc, feature, config, entities=None):
    return make_graph(doc['text'], doc['entities'] if entities is None else entities, feature, config['neighbors'], config['anchors'])

def run(args):
    output = Path(args.output).resolve()
    output.mkdir(parents=True, exist_ok=True)
    documents, audit = load_documents(args.dataset)
    save_json(output / 'data_audit.json', audit)
    print(json.dumps(audit['totals'], ensure_ascii=False, indent=2), flush=True)
    if args.audit_only:
        return
    if min(args.ner_epochs, args.relation_epochs, args.hidden, args.token_batch, args.encoder_batch, args.patience) < 1:
        raise ValueError('Epochs, dimensions, batches and patience must be positive')
    if args.neg_ratio <= 0 or min(args.neighbors, args.anchors) < 0 or (not 0 <= args.dropout < 1) or (args.lr <= 0):
        raise ValueError('Invalid sampling/model hyperparameters')
    if not args.variants or any((v not in {'semantic', 'rgcn'} for v in args.variants.split(','))):
        raise ValueError('Variants must be semantic,rgcn (or either one)')
    splits = split_documents(documents, output / 'split_manifest.json', args.seed)
    support = {name: dict(Counter((r['code'] for d in docs for r in d['relations']))) for name, docs in splits.items()}
    save_json(output / 'split_relation_support.json', support)
    save_json(output / 'environment.json', {name: importlib.metadata.version(name) for name in ['torch', 'torch-geometric', 'transformers', 'pyvi', 'numpy', 'scikit-learn']})
    device = torch.device(args.device or ('cuda' if torch.cuda.is_available() else 'cpu'))
    set_seed(args.seed)
    encoder = ContextEncoder(args.encoder, window=args.window, stride=args.stride, batch_size=args.encoder_batch, device=device, revision=args.revision, local_files_only=args.local_files_only)
    config = vars(args).copy()
    config['encoder_config'] = encoder.config
    config['thresholds'] = [0.35, 0.5, 0.65]
    save_json(output / 'run_config.json', config)
    features = {}
    for i, doc in enumerate(documents):
        encoded = encoder.encode(doc['text'], output / 'cache')
        features[doc['id']] = str(output / 'cache' / f'{encoded['cache_key']}.pt')
        print(f'Embedding {i + 1}/{len(documents)}: {doc['id']} ({len(encoded['offsets'])} units)', flush=True)
    del encoded, encoder
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    ner, ner_threshold = train_ner(splits, features, config, output, device)
    ner_test = evaluate_ner(ner, splits['test'], features, device, ner_threshold)
    save_json(output / 'ner_test.json', ner_test)
    graphs = {doc['id']: graph_for(doc, read_features(features[doc['id']]), config) for doc in documents}
    inferred_graphs = {}
    for doc in splits['test']:
        feature = read_features(features[doc['id']])
        inferred = predict_entities(ner, feature, doc['text'], device, ner_threshold)
        inferred_graphs[doc['id']] = graph_for(doc, feature, config, inferred)
    results = []
    for variant in args.variants.split(','):
        if variant not in {'semantic', 'rgcn'}:
            raise ValueError(f'Unknown variant: {variant}')
        relation, threshold = train_relations(splits, graphs, config, output, device, variant)
        oracle = evaluate_relations(relation, splits['test'], graphs, device, threshold)
        end_to_end = evaluate_relations(relation, splits['test'], inferred_graphs, device, threshold)
        save_json(output / f'test_{variant}.json', {'gold_entities': oracle, 'predicted_entities_end_to_end': end_to_end})
        for setting, metrics in [('gold_entities', oracle), ('predicted_entities_end_to_end', end_to_end)]:
            results.append({'variant': variant, 'setting': setting, **{k: v for k, v in metrics.items() if k != 'per_class'}})
    if results:
        with (output / 'comparison.csv').open('w', encoding='utf-8-sig', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=list(results[0]))
            writer.writeheader()
            writer.writerows(results)
    print(f'Finished. Actual measured results: {output}', flush=True)

def parser():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--dataset', required=True)
    p.add_argument('--output', default='output_results/raw_doc_v1')
    p.add_argument('--audit-only', action='store_true')
    p.add_argument('--encoder', default='vinai/phobert-base-v2')
    p.add_argument('--revision', default='main')
    p.add_argument('--local-files-only', action='store_true')
    p.add_argument('--device', default=None)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--window', type=int, default=256)
    p.add_argument('--stride', type=int, default=128)
    p.add_argument('--encoder-batch', type=int, default=8)
    p.add_argument('--token-batch', type=int, default=2048)
    p.add_argument('--ner-epochs', type=int, default=20)
    p.add_argument('--relation-epochs', type=int, default=30)
    p.add_argument('--patience', type=int, default=5)
    p.add_argument('--hidden', type=int, default=128)
    p.add_argument('--dropout', type=float, default=0.2)
    p.add_argument('--lr', type=float, default=0.001)
    p.add_argument('--neg-ratio', type=float, default=2.0)
    p.add_argument('--neighbors', type=int, default=8)
    p.add_argument('--anchors', type=int, default=12)
    p.add_argument('--variants', default='semantic,rgcn')
    return p

class Extractor:

    def __init__(self, model_dir, variant='rgcn', device=None, local_files_only=False):
        self.device = torch.device(device or ('cuda' if torch.cuda.is_available() else 'cpu'))
        model_dir = Path(model_dir)
        ner_ckpt = load_checkpoint(model_dir / 'checkpoints' / 'ner.pt', self.device)
        re_ckpt = load_checkpoint(model_dir / 'checkpoints' / f'relation_{variant}.pt', self.device)
        self.config = ner_ckpt['config']
        if re_ckpt['config'] != self.config or re_ckpt.get('variant') != variant:
            raise ValueError('NER and relation checkpoints must belong to the same run')
        self.ner = BoundaryNER(self.config['hidden'], self.config['dropout']).to(self.device)
        self.relation = LegalRGCN(self.config['hidden'], self.config['dropout'], variant == 'rgcn').to(self.device)
        self.ner.load_state_dict(ner_ckpt['model_state_dict'], strict=True)
        self.relation.load_state_dict(re_ckpt['model_state_dict'], strict=True)
        self.ner.eval()
        self.relation.eval()
        self.ner_threshold = ner_ckpt['threshold']
        self.relation_threshold = re_ckpt['threshold']
        ec = self.config['encoder_config']
        self.encoder = ContextEncoder(ec['model_name'], ec['window'], ec['stride'], self.config['encoder_batch'], self.device, ec['revision'], local_files_only)
        if self.encoder.config != ec:
            raise ValueError('Encoder/segmentation versions differ from training; use the recorded environment')

    @torch.no_grad()
    def predict(self, text):
        text = body_text(text)
        feature = self.encoder.encode(text)
        entities = predict_entities(self.ner, feature, text, self.device, self.ner_threshold)
        graph = graph_for({'text': text}, feature, self.config, entities)
        relations = []
        if graph['pairs']:
            h = self.relation.encode_nodes(graph['x'].to(self.device), graph['edge_index'].to(self.device), graph['edge_type'].to(self.device))
            for first in range(0, len(graph['pairs']), 2048):
                pairs = graph['pairs'][first:first + 2048]
                probabilities = self.relation.classify(h, torch.tensor(pairs, device=self.device).t()).sigmoid().cpu()
                for i, r in (probabilities >= self.relation_threshold).nonzero().tolist():
                    u, v = pairs[i]
                    relations.append({'from_id': entities[u]['id'], 'to_id': entities[v]['id'], 'code': ALL_RELATIONS[r], 'score': float(probabilities[i, r])})
        return {'full_text': text, 'entities': entities, 'relations': relations, 'note': 'Scores are uncalibrated model outputs; absent candidate pairs are not scored.'}

# RUN TRAINING
args = parser().parse_args([
    '--dataset', str(DATASET_DIR), '--output', str(OUTPUT_DIR),
    '--seed', str(SEED), '--encoder', ENCODER,
    '--ner-epochs', str(NER_EPOCHS), '--relation-epochs', str(RELATION_EPOCHS),
    '--encoder-batch', str(ENCODER_BATCH), '--variants', VARIANTS
]+(['--local-files-only'] if LOCAL_FILES_ONLY else []))
run(args)
print('🎉 Hoàn tất huấn luyện và đánh giá!')


### 📊 [BƯỚC 8] Vẽ Biểu đồ Quá trình Huấn luyện
Loss huấn luyện và F1 validation theo epoch. Điểm NER là exact-span/type;
điểm RE là Macro-F1 cố định 22 lớp, là hai tác vụ khác nhau.


In [ ]:
import json
import matplotlib.pyplot as plt

if not (OUTPUT_DIR/'ner_history.json').is_file():
    raise RuntimeError('Chạy xong Bước 7 trước khi vẽ biểu đồ.')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for name, metric, label in [('ner', 'val_f1', 'NER'), ('relation_semantic', 'val_f1_macro_22', 'RE semantic'), ('relation_rgcn', 'val_f1_macro_22', 'RE RGCN')]:
    path = OUTPUT_DIR/f'{name}_history.json'
    if not path.is_file():
        continue
    history = json.loads(path.read_text(encoding='utf-8'))
    epochs = [h['epoch'] for h in history]
    axes[0].plot(epochs, [h['loss'] for h in history], marker='o', label=label)
    axes[1].plot(epochs, [h[metric]*100 for h in history], marker='o', label=label)
axes[0].set(title='Training Loss', xlabel='Epoch', ylabel='Loss')
axes[1].set(title='Validation F1', xlabel='Epoch', ylabel='F1 (%)', ylim=(0, 100))
for ax in axes:
    ax.legend(); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'training_curves.png', dpi=180)
plt.show()


### 🎯 [BƯỚC 9] Đánh giá Test & Ma trận TP/FP/FN
Đọc hàng `predicted_entities_end_to_end` để đánh giá văn bản mới. `gold_entities` là RE khi
đã biết thực thể đúng. Vì đây là đa nhãn, vẽ TP/FP/FN từng quan hệ thay ma trận đơn nhãn.
Accuracy trên cặp ứng viên phải so với baseline luôn đoán không có quan hệ.


In [ ]:
import numpy as np

TEST_VARIANT = 'rgcn'  # Đổi thành 'semantic' nếu chỉ train baseline.
test_path = OUTPUT_DIR/f'test_{TEST_VARIANT}.json'
if not test_path.is_file():
    raise RuntimeError(f'Chưa có {test_path.name}; cần chạy xong Bước 7 với variant tương ứng.')
test_results = json.loads(test_path.read_text(encoding='utf-8'))
test_metrics = test_results['predicted_entities_end_to_end']
display(pd.DataFrame([{'setting': name, **{k:v for k,v in metrics.items() if k != 'per_class'}} for name, metrics in test_results.items()]))
class_report = pd.DataFrame(test_metrics['per_class']).T
class_report['fn'] = class_report['support'] - class_report['tp']
display(class_report)
matrix = class_report[['tp','fp','fn']].to_numpy(dtype=int)
fig, ax = plt.subplots(figsize=(15, 11))
ax.imshow(np.log1p(matrix), cmap='Blues', aspect='auto')
ax.set_xticks(range(3), ['TP', 'FP', 'FN'])
ax.set_yticks(range(len(class_report)), class_report.index, fontsize=8)
for i in range(len(class_report)):
    for j in range(3):
        ax.text(j, i, str(matrix[i,j]), ha='center', va='center', fontsize=8, color='darkred')
ax.set_title('End-to-end relation extraction: TP / FP / FN (color = log(1 + count))')
fig.tight_layout()
fig.savefig(OUTPUT_DIR/'confusion_matrix.png', dpi=180)
plt.show()
ner_test = json.loads((OUTPUT_DIR/'ner_test.json').read_text(encoding='utf-8'))
print('NER exact-span/type:', {k:v for k,v in ner_test.items() if k != 'per_type'})


### 📋 [BƯỚC 10] Bảng So sánh Kết quả Thực nghiệm
Chỉ đưa số liệu thật của các mô hình đã train. So sánh `semantic` và `rgcn` là ablation GNN;
chưa phải thí nghiệm bỏ sliding window. Macro chính cố định 22 lớp; điểm supported báo riêng.


In [ ]:
comparison_path = OUTPUT_DIR/'comparison.csv'
if not comparison_path.is_file():
    raise RuntimeError('Chưa có bảng kết quả. Chạy xong Bước 7 trước.')
comparison = pd.read_csv(comparison_path)
columns = ['variant', 'setting', 'accuracy_candidates_exact_set', 'precision_macro_22', 'f1_macro_22', 'f1_macro_supported', 'f1_micro_positive', 'candidate_recall', 'no_relation_baseline_accuracy']
table = comparison[columns].copy()
for column in columns[2:]:
    table[column] = (table[column]*100).round(2)
display(table)
table.to_csv(OUTPUT_DIR/'bang_so_sanh_nghiem_thu.csv', index=False, encoding='utf-8-sig')
(OUTPUT_DIR/'bang_so_sanh_nghiem_thu.md').write_text('# Kết quả thực đo (các chỉ số theo %)\n\n'+table.to_markdown(index=False), encoding='utf-8')
print(f'✅ Đã lưu bảng tại {OUTPUT_DIR}')


### 📥 [TÙY CHỌN] Đóng gói & Tải Kết quả
ZIP gồm checkpoint, báo cáo và hình ảnh; bỏ cache embedding lớn. Có thể chạy lại sau Bước 11
để thêm kết quả dự đoán. File ZIP được lưu trên Drive; chỉ tải về máy khi `DOWNLOAD_ZIP = True`.


In [ ]:
import zipfile

DOWNLOAD_ZIP = False
ZIP_PATH = OUTPUT_DIR.parent/(OUTPUT_DIR.name+'_results.zip')
with zipfile.ZipFile(ZIP_PATH,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        relative = path.relative_to(OUTPUT_DIR)
        if path.is_file() and relative.parts[0] != 'cache':
            archive.write(path,relative.as_posix())
print('✅ ZIP:', ZIP_PATH)
if DOWNLOAD_ZIP and IS_COLAB:
    from google.colab import files
    files.download(str(ZIP_PATH))


### 🔍 [BƯỚC 11] Test Thủ công & Thử nghiệm Tương tác
Nhập văn bản mới vào `NEW_TEXT`, hoặc đặt đường dẫn file `.txt` UTF-8 vào `NEW_TEXT_FILE`.
Không cần cung cấp thực thể hay đồ thị. Kết quả đầy đủ lưu ở `prediction.json`.
Score là đầu ra chưa hiệu chỉnh của mô hình, không phải độ chính xác đã được kiểm chứng.


In [ ]:
NEW_TEXT_FILE = None  # Ví dụ: '/content/drive/MyDrive/van_ban_moi.txt'
NEW_TEXT = """Điều 1. Phạm vi điều chỉnh
1. Văn bản này quy định quyền và nghĩa vụ của các bên trong hợp đồng.
2. Các bên có trách nhiệm thực hiện nghĩa vụ đã thỏa thuận.
"""

if not (OUTPUT_DIR/'checkpoints'/'relation_rgcn.pt').is_file():
    raise RuntimeError('Cần train xong RGCN ở Bước 7 trước khi test thủ công.')
if 'extractor' not in globals() or str(extractor_model_dir) != str(OUTPUT_DIR):
    extractor = Extractor(OUTPUT_DIR, local_files_only=LOCAL_FILES_ONLY)
    extractor_model_dir = OUTPUT_DIR
text = Path(NEW_TEXT_FILE).read_text(encoding='utf-8-sig') if NEW_TEXT_FILE else NEW_TEXT
prediction = extractor.predict(text)
save_json(OUTPUT_DIR/'prediction.json', prediction)
print(f"🔮 {len(prediction['entities'])} thực thể, {len(prediction['relations'])} quan hệ dự đoán.")
display(pd.DataFrame(prediction['entities']).head(30))
display(pd.DataFrame(prediction['relations']).head(30))
print('Đã lưu toàn bộ kết quả:', OUTPUT_DIR/'prediction.json')
